<div style="
background-color:#EAEAEA;
padding:15px;
border-left:5px solid #6C757D;
border-radius:6px;">

# Master's Thesis in Advanced Physics
---

This notebook is part of the **Master's Thesis (MSc Dissertation)**: **Fast Simulation of Neutrino Oscillations in Matter**.

**Author**  
Juan Ramon Diaz Santos <diazjuan@alumni.uv.es>

**Supervisors**  
Roberto Ruiz de Austri Bazan <rruiz@ific.uv.es>  
Michele Lucente <michele.lucente@unibo.it>

**Date**  
June 2026
</div>

# External - nuSQuIDS Interface Exploration
---

This notebook explores the optional external `nuSQuIDS` Python bindings used by TPeanuts as an independent reference backend. The workflow checks backend availability, configures a three-flavour solver, evaluates vacuum, solar, coherent-Earth, and Earth-atmosphere propagation examples, visualizes the resulting probabilities, and exports compact CSV reference tables.

The notebook intentionally uses the public helpers in `tpeanuts.external.nusquids.core`; it does not redefine nuSQuIDS setup or evolution helpers locally.

---

## Table of Contents

| # | Section |
|---|---------|
| [0](#0.-Theory-Background) | **Theory Background** - three-flavour evolution, nuSQuIDS solver, vacuum, solar, Earth, and EarthAtm propagation |
| [1](#1.-Libraries) | **Libraries** |
| [2](#2.-Paths-and-Configuration) | **Paths and Configuration** - paths, physical parameters, backend discovery |
| [3](#3.-Vacuum-Propagation) | **Vacuum Propagation** - single example, grid, and grid slice |
| [4](#4.-Solar-Propagation-Inside-the-Sun) | **Solar Propagation Inside the Sun** - density, production profiles, and SunASnu point propagation |
| [5](#5.-Coherent-Earth-Propagation) | **Coherent Earth Propagation** |
| [6](#6.-Earth-Atmosphere-Propagation) | **Earth-Atmosphere Propagation** |
| [7](#7.-Exported-Reference-Tables) | **Exported Reference Tables** |
| [8](#8.-Summary) | **Summary** |


## 0. Theory Background

### 0.1 Three-Flavour Oscillation Evolution

In the standard three-flavour framework, a neutrino flavour state evolves according to a Schrodinger-like equation

$$
i\frac{d}{dx}\nu_f(x) = H_f(x)\nu_f(x),
$$

where the flavour-basis Hamiltonian is the sum of the vacuum term and, when matter is present, an effective matter potential. In vacuum,

$$
H_{\rm vac} = \frac{1}{2E}\,U\,\operatorname{diag}(0,\Delta m^2_{21},\Delta m^2_{31})\,U^\dagger,
$$

with $U$ the PMNS matrix. The matter contribution for ordinary matter adds the charged-current Wolfenstein potential to the electron-flavour component,

$$
V_e(x)=\sqrt{2}\,G_F n_e(x),
$$

which modifies the effective eigenvalues and mixing angles during propagation. This matter formulation follows Wolfenstein's coherent forward-scattering treatment and the MSW development by Mikheyev and Smirnov.

---

### 0.2 nuSQuIDS as an External Reference Backend

`nuSQuIDS` is an external C++/Python solver for neutrino propagation. In this notebook it is not used through TPeanuts propagation code; instead, TPeanuts calls the installed `nuSQuIDS` bindings through `tpeanuts.external.nusquids.core`. The solver is configured with mixing angles, mass splittings, CP phase, and ODE tolerances, then evolved with `EvolveState()`.

For a pure initial flavour $\alpha$, the final probabilities are read as

$$
P_{\alpha\beta}(E,L) = |\langle \nu_\beta | \nu_\alpha(L)\rangle|^2,
$$

and are expected to satisfy unitarity, $\sum_\beta P_{\alpha\beta}=1$, up to numerical integration error. The grid checks in this notebook use that normalization as the first sanity test.

---

### 0.3 Vacuum Reference Case

Vacuum propagation is the cleanest external-reference mode because it contains no density-profile convention. For a fixed baseline $L$, the oscillation phase scales approximately as

$$
\phi_{ij} \propto \frac{\Delta m^2_{ij} L}{E},
$$

so the probability curves change most rapidly at low energy and long baseline. This makes the vacuum scan a useful diagnostic for unit conversions, flavour ordering, and solver configuration before adding matter effects.

---

### 0.4 EarthAtm Propagation and Zenith Geometry

`nuSQuIDS.EarthAtm()` combines an atmospheric production segment with Earth propagation. The trajectory is generated by `MakeTrackWithCosine(cos_zenith)`, so this notebook keeps the cosine-zenith convention explicit. Downward-going events near $\cos\theta_z=1$ have short matter paths, while upward-going events near $\cos\theta_z=-1$ cross a longer Earth chord and can show stronger matter-modified oscillation structure.

---

### References

- L. Wolfenstein, *Neutrino oscillations in matter*, Phys. Rev. D **17**, 2369 (1978).
- S. P. Mikheyev and A. Yu. Smirnov, *Resonance amplification of oscillations in matter and spectroscopy of solar neutrinos*, Sov. J. Nucl. Phys. **42**, 913 (1985).
- C. A. Arguelles, J. Salvado, and C. N. Weaver, *nuSQuIDS: A toolbox for neutrino propagation*, Comput. Phys. Commun. **277**, 108346 (2022), arXiv:2112.09122.
- Particle Data Group, *Review of Particle Physics*, neutrino mixing review, current standard three-flavour notation.


## 1. Libraries


## 2. Paths and Configuration

### 2.1 Paths

`load_notebook_config()` resolves the repository root, applies the shared matplotlib/numpy style used across the notebook suite, and creates output directories on demand.

The output directory follows the project convention: it is the global notebook output root plus the notebook path relative to `notebooks/`, excluding the notebook filename. For this notebook that relative directory is `external/nusquids`, so all figures and CSV files are written under `external/nusquids/`.


In [1]:
from __future__ import annotations

from dataclasses import asdict

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tpeanuts.notebooks.notebookConfig import load_notebook_config
from tpeanuts.notebooks.notebooks_helper import FLAVOUR_NAMES, save_and_show, to_numpy
from tpeanuts.external.nusquids.core import (
    NuSQuIDSConfig,
    NuSQuIDSError,
    is_available,
    make_solar_track,
    probability_atmosphere,
    probability_earth,
    probability_grid_vacuum,
    probability_solar_point,
    probability_vacuum,
    require_nusquids,
    sun_asnu_radius,
    sun_asnu_track_fraction,
)
from tpeanuts.medium.solar.profile import SolarProfile
from tpeanuts.util.context import RuntimeContext
from tpeanuts.util.math import numpy_trapezoid


ImportError: cannot import name 'atmosphere_mass_density_profile_nusquids' from 'tpeanuts.external.nusquids.density' (G:\Mi unidad\03.Codigo\034.TFM.UV\Tpeanuts\external\nusquids\density.py)

In [ ]:
config = load_notebook_config()
ctx = RuntimeContext.resolve(config.device, config.dtype)
OUTPUT_DIR = config.output_dir("external", "nusquids")
SHOW = config.show_plots

print(f"Package dir : {config.package_dir}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Device      : {ctx.device}   dtype: {ctx.dtype}")
print(f"Show plots  : {SHOW}")


### 2.2 Configuration

The physical setup is a standard three-flavour oscillation benchmark. `NuSQuIDSConfig` stores the mixing angles, mass splittings, CP phase, and numerical tolerances passed to the external solver. Energies are expressed in GeV for vacuum/Earth scans and in MeV for solar-neutrino scans.

| Parameter | Value | Description |
|-----------|-------|-------------|
| Backend | `nuSQuIDS` Python bindings | Optional external C++ solver used as a reference |
| Flavours | `nue`, `numu`, `nutau` | Probability output ordering |
| $\theta_{12},\theta_{13},\theta_{23}$ | `NuSQuIDSConfig` defaults | Three-flavour PMNS mixing angles in radians |
| $\delta_{CP}$ | `NuSQuIDSConfig` default | Dirac CP phase in radians |
| $\Delta m^2_{21}$ | `NuSQuIDSConfig` default | Solar mass splitting in eV$^2$ |
| $\Delta m^2_{3l}$ | `NuSQuIDSConfig` default | Atmospheric mass splitting in eV$^2$ |
| `rel_error`, `abs_error` | $10^{-10}$, $10^{-12}$ | ODE integration tolerances |
| `h_max_km` | 500 km | Maximum internal step size, when supported |
| Vacuum energy grid | 20 points, 0.5-100 GeV | Logarithmic grid for vacuum tables |
| Solar energy grid | 20 points, 0.2-20 MeV | Solar propagation window |
| Earth scans | 41 cosine-zenith points at 10 GeV | Geometry/matter-effect diagnostics |


In [ ]:
NSQ_CONFIG = NuSQuIDSConfig(
    rel_error=1.0e-10,
    abs_error=1.0e-12,
    h_max_km=500.0,
)

ENERGIES_GEV = np.logspace(np.log10(0.5), np.log10(100.0), 5)
BASELINES_KM = np.array([20.0, 100.0, 500.0, 1000.0, 5000.0, 10000.0], dtype=float)
INITIAL_FLAVOURS = tuple(FLAVOUR_NAMES)

PLOT_BASELINE_KM = 1000.0
PLOT_INITIAL_FLAVOUR = "numu"
EARTH_ENERGY_GEV = 10.0
PLOT_COS_ZENITH = np.linspace(-1.0, 1.0, 41)

SOLAR_PROFILE = SolarProfile.default(context=ctx)
SOLAR_SOURCES = ("pp", "pep", "hep", "7Be", "8B", "13N", "15O", "17F")
SOLAR_ENERGIES_MEV = np.linspace(0.2, 20.0, 20)
SOLAR_POINT_RADII = (0.0, 0.05, 0.20, 0.50)

print("nuSQuIDS physical/numerical configuration")
display(pd.DataFrame([asdict(NSQ_CONFIG)]))
print(f"Vacuum energy grid points : {len(ENERGIES_GEV)}")
print(f"Solar profile points      : {len(SOLAR_PROFILE.radius)}")


### 2.3 Optional Backend Discovery

This block checks whether the optional `nuSQuIDS` bindings are importable. The adapter tries the common Python module names used by different installations and reports a clear error if none are available. No notebook-local helper functions are defined here; reusable operations live in `tpeanuts.external.nusquids.core`.

**Expected results:** if `nuSQuIDS` is installed in the active environment, the cell prints the resolved module name and `True`; otherwise, downstream nuSQuIDS numerical cells are skipped gracefully.


In [ ]:
try:
    nsq_module = require_nusquids()
    print("nuSQuIDS module:", nsq_module.__name__)
    print("nuSQuIDS available:", True)
except NuSQuIDSError as exc:
    nsq_module = None
    print(exc)

NSQ_AVAILABLE = is_available()
try:
    R_SUN_NUSQUIDS = sun_asnu_radius() if NSQ_AVAILABLE else None
except NuSQuIDSError:
    R_SUN_NUSQUIDS = None

print("Adapter availability:", NSQ_AVAILABLE)
print("SunASnu radius available:", R_SUN_NUSQUIDS is not None)


## 3. Vacuum Propagation

Vacuum propagation is the cleanest reference mode because no density profile or geometry convention enters the calculation.


### 3.1 Vacuum Propagation Example

A single $\nu_\mu$ state is propagated over a 1000 km vacuum baseline at 3 GeV. The output table shows the final probabilities ordered as $\nu_e$, $\nu_\mu$, and $\nu_\tau$.

**Expected results:** the three probabilities should be finite, lie between 0 and 1, and sum to unity up to the configured solver tolerance. This is the minimal check that the backend, units, flavour ordering, and solver configuration are consistent.


In [ ]:
if NSQ_AVAILABLE:
    vacuum_example_probs = probability_vacuum(
        E_GeV=3.0,
        baseline_km=1000.0,
        initial_flavour="numu",
        config=NSQ_CONFIG,
    )
    vacuum_example = pd.DataFrame([dict(zip(FLAVOUR_NAMES, vacuum_example_probs))])
    vacuum_example["probability_sum"] = vacuum_example[list(FLAVOUR_NAMES)].sum(axis=1)
    vacuum_example["unitarity_error"] = np.abs(vacuum_example["probability_sum"] - 1.0)
    display(vacuum_example)
else:
    vacuum_example_probs = np.full(len(FLAVOUR_NAMES), np.nan)
    print("Skipping vacuum example because nuSQuIDS is unavailable.")


### 3.2 Vacuum Probability Grid

The compact vacuum grid scans 20 energies, several baselines, and all initial flavours. It is useful as a reusable reference table for later validation notebooks because no matter-density model enters the calculation.

**Expected results:** probability normalization errors should remain close to machine/integrator precision. At fixed baseline, the flavour probabilities should vary with energy because the vacuum phases scale as $L/E$.


In [ ]:
if NSQ_AVAILABLE:
    vacuum_table = probability_grid_vacuum(
        E_GeV=ENERGIES_GEV,
        baseline_km=BASELINES_KM,
        initial_flavours=INITIAL_FLAVOURS,
        antinu=False,
        config=NSQ_CONFIG,
    )
    vacuum_table["probability_sum"] = vacuum_table[["P_nue", "P_numu", "P_nutau"]].sum(axis=1)
    vacuum_table["normalization_error"] = np.abs(vacuum_table["probability_sum"] - 1.0)
    display(vacuum_table.head(12))
    display(vacuum_table.groupby("initial_flavour")["normalization_error"].agg(["max", "median"]))
else:
    vacuum_table = pd.DataFrame()
    print("Skipping vacuum grid because nuSQuIDS is unavailable.")


### 3.3 Vacuum Grid Slice

The figure below extracts the $L=1000$ km, initial-$\nu_\mu$ slice from the vacuum grid and plots the three final-flavour probabilities versus energy.

**Expected results:** $P_{\mu\mu}$ remains dominant in parts of the scan, while $P_{\mu e}$ and $P_{\mu\tau}$ oscillate with energy. The curves should stay inside the physical interval $[0,1]$.


In [ ]:
if not vacuum_table.empty:
    plot_df = vacuum_table[
        (vacuum_table["baseline_km"] == PLOT_BASELINE_KM)
        & (vacuum_table["initial_flavour"] == PLOT_INITIAL_FLAVOUR)
    ].sort_values("E_GeV")

    fig, ax = plt.subplots(figsize=(8.0, 4.6))
    for flavour in FLAVOUR_NAMES:
        ax.semilogx(plot_df["E_GeV"], plot_df[f"P_{flavour}"], marker=".", label=flavour)
    ax.set_xlabel("Energy [GeV]")
    ax.set_ylabel("Probability")
    ax.set_ylim(-0.03, 1.03)
    ax.set_title(f"nuSQuIDS vacuum probabilities, {PLOT_INITIAL_FLAVOUR}, L={PLOT_BASELINE_KM:g} km")
    ax.legend()
    fig.tight_layout()
    save_and_show("eda1_fig33_vacuum_grid_slice.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW)
else:
    print("Skipping vacuum figure because the vacuum table is empty.")


## 4. Solar Propagation Inside the Sun

This section inspects the solar inputs used around the `SunASnu` reference workflow. The `nuSQuIDS` propagation itself is performed by `probability_solar_point`, while the displayed density and production profiles come from the tabulated `SolarProfile` object used to build source-weighted solar validation grids.


### 4.1 Solar Density Profile

The plot compares the TPeanuts solar electron-density profile with the density profile sampled directly from the `nuSQuIDS.SunASnu` body. The TPeanuts curve is kept as a reference because it is used by the source-weighted validation workflow, while the nuSQuIDS curve is the actual body profile used by `SunASnu` propagation.

Important implementation detail: the physical solar radius $\rho_{\rm solar}=r/R_\odot$ is not the same as the normalized `SunASnu.Track` coordinate. For the radial `SunASnu` trajectory, the solar interior from centre to surface is sampled over the second half of the track coordinate:

$$
x_{\rm nsq}/R_{\rm SunASnu}=0.5+0.5\rho_{\rm solar}.
$$

Thus $\rho_{\rm solar}=0$ maps to the solar centre at `x/R = 0.5`, and $\rho_{\rm solar}=1$ maps to the surface at `x/R = 1`. Sampling `density(track)` at `x/R = rho_solar` incorrectly probes the wrong part of the trajectory and can make the apparent central density tend to zero.

For `SunASnu`, the body returns mass density in g/cm$^3$ and electron fraction $Y_e$ along a track. The notebook displays $\rho Y_e$, which is proportional to electron density and directly comparable in shape, though not necessarily in absolute units, to the TPeanuts mol/cm$^3$ profile.

**Expected results:** both profiles should be largest in the solar core and decrease steeply toward the surface. They need not coincide point by point because TPeanuts and nuSQuIDS can use different built-in standard solar model tables and units.


In [ ]:
radius_np = to_numpy(SOLAR_PROFILE.radius)
density_np = to_numpy(SOLAR_PROFILE.density)

nsq_solar_density_np = np.full_like(radius_np, np.nan, dtype=float)
nsq_solar_ye_np = np.full_like(radius_np, np.nan, dtype=float)
nsq_solar_density_available = False

if NSQ_AVAILABLE and R_SUN_NUSQUIDS is not None:
    try:
        nsq = require_nusquids()
        nsq_sun = nsq.SunASnu()
        for idx, rho_solar in enumerate(radius_np):
            x_nsq_norm = sun_asnu_track_fraction(float(rho_solar))
            track = make_solar_track(float(rho_solar), nsq=nsq)
            # density(track) is evaluated at track.GetX(), so set the current
            # coordinate explicitly to the mapped SunASnu track coordinate.
            track.SetX(x_nsq_norm * R_SUN_NUSQUIDS)
            nsq_solar_density_np[idx] = float(nsq_sun.density(track))
            nsq_solar_ye_np[idx] = float(nsq_sun.ye(track))
        nsq_solar_density_available = np.isfinite(nsq_solar_density_np).any()
    except Exception as exc:
        print("Could not sample nuSQuIDS SunASnu density profile:", exc)

fig, ax_left = plt.subplots(figsize=(8.4, 4.8))
ax_left.semilogy(radius_np, np.maximum(density_np, 1e-300), color="C0", lw=1.8, label="TPeanuts $n_e$")
ax_left.set_xlabel(r"Solar radius $r/R_\odot$")
ax_left.set_ylabel(r"TPeanuts $n_e$ [mol cm$^{-3}$]", color="C0")
ax_left.tick_params(axis="y", labelcolor="C0")
ax_left.set_ylim(1e-7, 2e2)

if nsq_solar_density_available:
    ax_right = ax_left.twinx()
    nsq_rho_ye_np = nsq_solar_density_np * nsq_solar_ye_np
    ax_right.semilogy(radius_np, np.maximum(nsq_rho_ye_np, 1e-300), color="C3", lw=1.5, ls="--", label=r"nuSQuIDS $\rho Y_e$")
    ax_right.set_ylabel(r"nuSQuIDS $\rho Y_e$ [g cm$^{-3}$]", color="C3")
    ax_right.tick_params(axis="y", labelcolor="C3")
    lines = ax_left.get_lines() + ax_right.get_lines()
    ax_left.legend(lines, [line.get_label() for line in lines], fontsize=8)
    ax_right.set_ylim(1e-7, 2e2)
else:
    ax_left.text(
        0.52, 0.18,
        "nuSQuIDS SunASnu density\nnot available in this environment",
        transform=ax_left.transAxes,
        ha="center",
        va="center",
        fontsize=9,
        bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": "0.7"},
    )
    ax_left.legend(fontsize=8)

ax_left.set_title("Solar density profile: TPeanuts reference vs nuSQuIDS SunASnu")
fig.tight_layout()
save_and_show("eda1_fig41_solar_density_tpeanuts_vs_nusquids.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW)

print(f"TPeanuts density(core)    = {density_np[0]:.4e} mol cm^-3")
print(f"TPeanuts density(surface) = {density_np[-1]:.4e} mol cm^-3")
if nsq_solar_density_available:
    print(f"nuSQuIDS rho(core)        = {nsq_solar_density_np[0]:.4e} g cm^-3")
    print(f"nuSQuIDS Ye(core)         = {nsq_solar_ye_np[0]:.4e}")
    print(f"nuSQuIDS rho*Ye(core)     = {(nsq_solar_density_np[0] * nsq_solar_ye_np[0]):.4e} g cm^-3")


### 4.2 Solar Production Profiles

The left panel shows the TPeanuts source-production profiles (`pp`, `pep`, `hep`, `$^7$Be`, `$^8$B`, CNO sources). These are the profiles used by TPeanuts when building source-averaged solar probabilities.

The right panel is reserved for the corresponding profiles exposed by `nuSQuIDS.SunASnu`. In the current nuSQuIDS public body API, `SunASnu` exposes the solar density/composition profile used for propagation, but it does not expose source-production fractions such as `pp` or `$^8$B`; point propagation receives an explicit production radius instead. The notebook therefore reports this explicitly rather than substituting the TPeanuts profiles.

**Expected results:** TPeanuts profiles peak in the solar core with different widths by source. The nuSQuIDS panel should either show the profiles if a future binding exposes them, or state that no source-production profiles are available from `SunASnu`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0), sharey=True)
ax_tp, ax_nsq = axes

for source in SOLAR_SOURCES:
    if source not in SOLAR_PROFILE.fractions:
        continue
    values = to_numpy(SOLAR_PROFILE.fractions[source])
    norm = numpy_trapezoid(values, radius_np)
    if np.isfinite(norm) and norm > 0.0:
        values = values / norm
    ax_tp.plot(radius_np, values, label=source)

ax_tp.set_xlabel(r"Solar radius $r/R_\odot$")
ax_tp.set_ylabel("Normalized production profile")
ax_tp.set_title("TPeanuts source profiles")
ax_tp.set_xlim(0.0, 0.45)
ax_tp.legend(ncol=2, fontsize=8)

nsq_source_profiles_available = False
nsq_source_profile_names = []
if NSQ_AVAILABLE:
    try:
        nsq = require_nusquids()
        nsq_sun = nsq.SunASnu()
        candidate_names = [name for name in dir(nsq_sun) if any(token in name.lower() for token in ("source", "production", "flux", "fraction"))]
        nsq_source_profile_names = candidate_names
        # The public SunASnu body currently exposes density/ye/composition, not pp/8B production fractions.
        nsq_source_profiles_available = False
    except Exception as exc:
        print("Could not inspect nuSQuIDS SunASnu production-profile API:", exc)

ax_nsq.set_xlabel(r"Solar radius $r/R_\odot$")
ax_nsq.set_title("nuSQuIDS SunASnu source profiles")
ax_nsq.set_xlim(0.0, 0.45)

if nsq_source_profiles_available:
    ax_nsq.legend(ncol=2, fontsize=8)
else:
    msg = "SunASnu does not expose\npp / 8B / CNO production\nprofiles via the public body API."
    if nsq_source_profile_names:
        msg += "\n\nRelated attributes found:\n" + ", ".join(nsq_source_profile_names[:4])
    elif not NSQ_AVAILABLE:
        msg = "nuSQuIDS is not available\nin this environment."
    ax_nsq.text(
        0.5, 0.5,
        msg,
        transform=ax_nsq.transAxes,
        ha="center",
        va="center",
        fontsize=9,
        bbox={"boxstyle": "round,pad=0.35", "fc": "white", "ec": "0.7"},
    )

fig.suptitle("Solar production profiles: TPeanuts reference and nuSQuIDS availability", y=1.02)
fig.tight_layout()
save_and_show("eda1_fig42_solar_production_profiles_tpeanuts_vs_nusquids.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW)

if nsq_source_profile_names:
    print("nuSQuIDS SunASnu related attributes:", nsq_source_profile_names)
else:
    print("nuSQuIDS SunASnu production profiles are not exposed by the public body API in this environment/API.")


### 4.3 SunASnu Point Propagation

For a small set of fixed production radii, this scan calls `probability_solar_point` and reads the decohered electron-neutrino survival probability after propagation through `SunASnu`.

**Expected results:** if `nuSQuIDS` is available, core-produced high-energy neutrinos should show stronger MSW suppression than neutrinos produced near the outer solar region. If `nuSQuIDS` is unavailable, this diagnostic is skipped without failing the notebook.


In [ ]:
if NSQ_AVAILABLE:
    solar_rows = []
    print('Starting simulation in ',SOLAR_POINT_RADII*SOLAR_ENERGIES_MEV, '(radius x energy) grid points')
    for r0 in SOLAR_POINT_RADII:
        for E_MeV in SOLAR_ENERGIES_MEV:
            print('.')
            probs = probability_solar_point(E_MeV=float(E_MeV), r0=float(r0), config=NSQ_CONFIG)
            solar_rows.append({"r0": float(r0), "E_MeV": float(E_MeV), "P_nue": probs[0], "P_numu": probs[1], "P_nutau": probs[2]})
    solar_point_table = pd.DataFrame(solar_rows)
    display(solar_point_table.head())

    fig, ax = plt.subplots(figsize=(8.0, 4.6))
    for r0, sub in solar_point_table.groupby("r0"):
        ax.plot(sub["E_MeV"], sub["P_nue"], marker="o", label=rf"$r_0={r0:.2f}R_\odot$")
    ax.set_xlabel("Energy [MeV]")
    ax.set_ylabel(r"$P_{ee}$")
    ax.set_ylim(0.0, 1.0)
    ax.set_title("SunASnu point propagation: electron-neutrino survival")
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_and_show("eda1_fig43_solar_point_propagation.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW)
else:
    solar_point_table = pd.DataFrame()
    print("Skipping SunASnu solar propagation because nuSQuIDS is unavailable.")


## 5. Coherent Earth Propagation

This section uses the `nuSQuIDS` Earth body through `probability_earth`. Unlike the solar observable, the final flavour probabilities are read coherently from the evolved state for the specified Earth trajectory.

**Expected results:** upward-going trajectories with negative $\cos\theta_z$ cross longer Earth chords and can show stronger matter-modified flavour conversion than downward-going trajectories.


In [ ]:
if NSQ_AVAILABLE:
    earth_coherent_rows = []
    for cz in PLOT_COS_ZENITH:
        probs = probability_earth(
            E_GeV=EARTH_ENERGY_GEV,
            cos_zenith=float(cz),
            initial_flavour="numu",
            config=NSQ_CONFIG,
        )
        earth_coherent_rows.append({"cos_zenith": float(cz), "P_nue": probs[0], "P_numu": probs[1], "P_nutau": probs[2]})
    earth_coherent_table = pd.DataFrame(earth_coherent_rows)
    earth_coherent_table["probability_sum"] = earth_coherent_table[["P_nue", "P_numu", "P_nutau"]].sum(axis=1)
    earth_coherent_table["normalization_error"] = np.abs(earth_coherent_table["probability_sum"] - 1.0)
    display(earth_coherent_table.head())
    display(earth_coherent_table["normalization_error"].agg(["max", "median"]).to_frame().T)

    fig, ax = plt.subplots(figsize=(8.0, 4.6))
    for flavour in FLAVOUR_NAMES:
        ax.plot(earth_coherent_table["cos_zenith"], earth_coherent_table[f"P_{flavour}"], label=flavour)
    ax.set_xlabel(r"$\cos\theta_z$")
    ax.set_ylabel("Probability")
    ax.set_ylim(-0.03, 1.03)
    ax.set_title(r"nuSQuIDS coherent Earth scan, $\nu_\mu$, $E=10$ GeV")
    ax.legend()
    fig.tight_layout()
    save_and_show("eda1_fig50_coherent_earth_scan.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW)
else:
    earth_coherent_table = pd.DataFrame()
    print("Skipping coherent Earth scan because nuSQuIDS is unavailable.")


## 6. Earth-Atmosphere Propagation

This scan uses `probability_atmosphere`, which wraps `nuSQuIDS.EarthAtm()` and `MakeTrackWithCosine(cos_zenith)`. The initial state is $\nu_\mu$ at 10 GeV, and the scan covers the full zenith-cosine interval.

**Expected results:** downward-going trajectories near $\cos\theta_z=1$ have shorter matter paths, while upward-going trajectories near $\cos\theta_z=-1$ cross longer Earth chords. The probabilities should remain normalized and can show stronger flavour conversion for long paths.


In [ ]:
if NSQ_AVAILABLE:
    rows = []
    for cz in PLOT_COS_ZENITH:
        probs = probability_atmosphere(
            E_GeV=EARTH_ENERGY_GEV,
            cos_zenith=float(cz),
            initial_flavour="numu",
            config=NSQ_CONFIG,
        )
        rows.append({"cos_zenith": float(cz), "P_nue": probs[0], "P_numu": probs[1], "P_nutau": probs[2]})
    earth_table = pd.DataFrame(rows)
    earth_table["probability_sum"] = earth_table[["P_nue", "P_numu", "P_nutau"]].sum(axis=1)
    earth_table["normalization_error"] = np.abs(earth_table["probability_sum"] - 1.0)
    display(earth_table.head())
    display(earth_table["normalization_error"].agg(["max", "median"]).to_frame().T)
else:
    earth_table = pd.DataFrame()
    print("Skipping EarthAtm scan because nuSQuIDS is unavailable.")


### 6.1 EarthAtm Zenith Scan

The figure shows the final flavour probabilities as a function of $\cos\theta_z$ for the 10 GeV atmospheric reference case.

**Expected results:** the curves should vary smoothly with zenith angle. Any sharp discontinuity would indicate a geometry convention issue, a backend problem, or an overly coarse numerical configuration.


In [ ]:
if not earth_table.empty:
    fig, ax = plt.subplots(figsize=(8.0, 4.6))
    for flavour in FLAVOUR_NAMES:
        ax.plot(earth_table["cos_zenith"], earth_table[f"P_{flavour}"], label=flavour)
    ax.set_xlabel(r"$\cos\theta_z$")
    ax.set_ylabel("Probability")
    ax.set_ylim(-0.03, 1.03)
    ax.set_title(r"nuSQuIDS EarthAtm scan, $\nu_\mu$, $E=10$ GeV")
    ax.legend()
    fig.tight_layout()
    save_and_show("eda1_fig61_earthatm_coszenith_scan.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW)
else:
    print("Skipping EarthAtm figure because the EarthAtm table is empty.")


## 7. Exported Reference Tables

The generated tables are saved as CSV files in the notebook output directory. These files are intended as lightweight external references for validation notebooks and regression tests.

**Expected results:** when `nuSQuIDS` is available, the vacuum grid, solar point-propagation table, coherent Earth scan, and EarthAtm scan are written. If the backend is unavailable, only the sections independent of `nuSQuIDS` produce figures and no external CSV tables are produced.


In [ ]:
saved_paths = []

if not vacuum_table.empty:
    path = OUTPUT_DIR / "nusquids_vacuum_grid.csv"
    vacuum_table.to_csv(path, index=False)
    saved_paths.append(path)

if not solar_point_table.empty:
    path = OUTPUT_DIR / "nusquids_solar_point_propagation.csv"
    solar_point_table.to_csv(path, index=False)
    saved_paths.append(path)

if not earth_coherent_table.empty:
    path = OUTPUT_DIR / "nusquids_coherent_earth_scan.csv"
    earth_coherent_table.to_csv(path, index=False)
    saved_paths.append(path)

if not earth_table.empty:
    path = OUTPUT_DIR / "nusquids_earthatm_coszenith_scan.csv"
    earth_table.to_csv(path, index=False)
    saved_paths.append(path)

if saved_paths:
    for path in saved_paths:
        print(path)
else:
    print("No external reference tables were exported.")


## 8. Summary

| Check | Observable | Interpretation |
|-------|------------|----------------|
| Backend discovery | Importable `nuSQuIDS` module | Confirms that the optional external reference backend can be used in this environment |
| Vacuum example/grid | $P_{\mu\beta}$ and normalization over 20 energies | Tests unit conversions, flavour ordering, and solver stability without density-model assumptions |
| Solar density/profile plots | $n_e(r)$ and source production shapes | Documents the solar profile used around the source-weighted SunASnu validation workflow |
| Solar point propagation | $P_{ee}(E,r_0)$ through `SunASnu` | Exercises the decohered solar reference wrapper when `nuSQuIDS` is available |
| Coherent Earth propagation | $P_{\mu\beta}(\cos\theta_z)$ through the Earth body | Tests coherent matter propagation through Earth-like trajectories |
| EarthAtm scan | $P_{\mu\beta}(\cos\theta_z)$ through `EarthAtm` | Exercises the atmosphere-plus-Earth geometry interface |
| CSV export | Vacuum, solar, Earth, and EarthAtm tables | Provides compact external data products for later validation notebooks |

**Physical interpretation:**

1. Vacuum propagation isolates the PMNS phase structure: the probability pattern depends on $L/E$ and should conserve total probability independently of any matter model.
2. Solar propagation is treated differently from detector-scale coherent propagation: the point evolution through `SunASnu` is converted into a decohered solar observable when mass-basis information is available.
3. Coherent Earth and EarthAtm scans keep phase information in the final flavour probabilities, so they are useful diagnostics for geometry conventions and matter-density handling in the external backend.


In [ ]:
print("Summary - External nuSQuIDS 1: Interface Exploration")
print("-" * 68)
print(f"nuSQuIDS available             : {NSQ_AVAILABLE}")
print(f"Output directory               : {OUTPUT_DIR}")
print(f"Vacuum energy grid points      : {len(ENERGIES_GEV)}")

if not vacuum_table.empty:
    print(f"Vacuum rows                    : {len(vacuum_table)}")
    print(f"Vacuum max norm. error         : {vacuum_table['normalization_error'].max():.3e}")
else:
    print("Vacuum rows                    : 0")

print(f"Solar profile sources          : {', '.join(SOLAR_SOURCES)}")
if not solar_point_table.empty:
    print(f"Solar point rows               : {len(solar_point_table)}")
else:
    print("Solar point rows               : 0")

if not earth_coherent_table.empty:
    print(f"Coherent Earth rows            : {len(earth_coherent_table)}")
    print(f"Coherent Earth max norm. error : {earth_coherent_table['normalization_error'].max():.3e}")
else:
    print("Coherent Earth rows            : 0")

if not earth_table.empty:
    print(f"EarthAtm rows                  : {len(earth_table)}")
    print(f"EarthAtm max norm. error       : {earth_table['normalization_error'].max():.3e}")
else:
    print("EarthAtm rows                  : 0")

print("Exported tables                :", len(saved_paths))
